# Neuromodulatory tuning of attentional sampling

This notebook continues the active-vision tutorial by asking a more specific question:

> **How should an event-driven attentional system balance exploration and exploitation?**

The sensory input is kept fixed: the same drift-generated RGB sequence is converted into the same DVS event stream, and the same proto-object saliency computation is used throughout. We then vary only the **fixation-selection policy**.

We proceed with two experiments:

1. **Softmax versus Argmax.** We compare stochastic exploratory sampling with deterministic winner-take-all selection.
   - **Softmax** is a stochastic fixation policy. Its inverse-temperature parameter $\beta$ controls how strongly saliency biases the selection of the next fixation.
   - **Argmax** is a deterministic winner-take-all control. It always selects the maximally salient location and has no $\beta$ parameter.

2. **Softmax gain sweep.** We keep the policy stochastic and vary the inverse-temperature parameter $\beta$ to test whether the computational system exhibits the expected exploration--exploitation trade-off observed in biological systems, and whether an intermediate gain can outperform both highly diffuse and highly selective sampling.

For every object and stochastic Softmax condition, we use **five seeds**. Seed-level runs constitute repeated measurements of the same visual stimulus and are therefore averaged within each object before statistical inference is performed across objects. We quantify performance using the following readouts:

- **explored object area**: the fraction of event-defined object cells visited by the fixation sequence;
- **normalised fixation entropy**: the extent to which fixations are distributed across those object cells.

## 1. Setup

Just like the original tutorial notebook.


In [ ]:
import sys
from pathlib import Path

repo = Path.cwd()
sys.path.insert(0, str(repo / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, HTML

from scene.render_offline import (
    render_camera_motion_sequence,
    frames_to_gif,
    frames_to_events_npy_and_gif,
)

from attention.attention import run_attention

from analysis_helpers import (
    display_gif_grid,
    compute_attention_exploration_metrics,
    plot_attention_exploration,
    plot_rgb_dvs_snapshot_grid,
    plot_attentional_gain_concept,
)

try:
    import bpy
    print("Blender Python available:", bpy.app.version_string)
except Exception as e:
    print("Warning: bpy could not be imported.")
    print(e)

Now we set the object roots and parameters

In [ ]:
OBJECT_SPECS = [
    {
        "name": "airplane",
        "relative_path": "data/airplane_010.blend",
    },
    {
        "name": "apple",
        "relative_path": "data/apple_020.blend",
    },
    {
        "name": "hammer",
        "relative_path": "data/hammer_023.blend",
    },
    {
        "name": "bus",
        "relative_path": "data/bus_012.blend",
    },
    {
        "name": "laptop",
        "relative_path": "data/laptop_036.blend",
    },
    {
        "name": "cat",
        "relative_path": "data/cat_064.blend",
    },
]

OUTPUT_ROOT = repo / "data" / "renders" / "nm_beta_sampling"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

RESOLUTION = 256
FPS = 1000
NUM_FRAMES = 500
WINDOW_PERIOD_MS = 10.0

RENDER_PARAMS = dict(
    resolution=RESOLUTION,
    samples=1,
    use_gpu=True,
    object_target_size=0.45,
    object_azimuth_deg=315.0,
    object_elevation_deg=30.0,
    camera_position=(0.0, -0.25, 2.0),
    camera_target=(0.0, 0.0, 0.0),
    focal_length=50.0,
    sensor_width_mm=32.0,
    light_location=(0.0, 0.0, 5.0),
    light_size=10.0,
    light_strength=50.0,
    drift_sigma_deg=(0.10, 0.09),
)

EVENT_PARAMS = dict(
    fps=FPS,
    th_pos=0.15,
    th_neg=0.15,
    th_noise=0.05,
    lat=500,
    tau=300,
    jit=100,
    bgnp=0.001,
    bgnn=0.001,
    ref=40,
    skip_frames=0,
    gif_fps=20,
    gif_window_us=1000,
    loop=0,
)

ATTENTION_PARAMS_BASE = {
    "saliency_backend": "lif_vm",
    "num_pyr": 4,
    "lif_thetas": np.arange(0.0, 2.0 * np.pi, np.pi / 4),
    "lif_tau_mem": 0.3,
    "lif_size_krn": 16,
    "lif_rho": 0.1,
    "lif_r0": 14,
    "lif_thick": 3.0,
    "lif_offset": (0, 0),
    "lif_filter_resize_perc": 1.0,
    "lif_stride": 1,
    "lif_out_ch": 1,
    "lif_device": "auto",
    "lif_stateful": True,
}

# Experiment 1
SOFTMAX_COMPARE_BETA = 0.5

# Experiment 2
BETA_SWEEP = [0.5, 1.0, 5.0, 10.0, 25.0, 50.0]

# Five stochastic repetitions
SEEDS = [0, 1, 2, 3, 4]

GRID_PER = 0.1
NOISE_THRESH = 100
EXCLUDE_INITIAL_FIXATION = True

# Consistent figure colours
COLORS = {
    "exploration": "navy",
    "wta": "crimson",
    "paired": "slategray",
}


## 2. Generate RGB motion and DVS event streams for all objects

This cell renders the drift-based RGB sequence for each object, converts it into a gif, then converts the sequence into events and builds the DVS gif.

If outputs already exist, the same cell can be reused and the files will simply be overwritten.

In [ ]:
def prepare_single_object_stream(spec, seed=0):
    object_name = spec["name"]
    object_path = repo / spec["relative_path"]

    if not object_path.exists():
        raise FileNotFoundError(
            f"Object not found: {object_path}"
        )

    obj_root = (
        OUTPUT_ROOT
        / object_name
    )

    sequence_dir = (
        obj_root
        / "motion_sequence"
    )

    events_dir = (
        sequence_dir
        / "events"
    )

    seq = render_camera_motion_sequence(
        object_path=object_path,
        output_dir=sequence_dir,
        num_frames=NUM_FRAMES,
        fps=FPS,
        seed=seed,
        **RENDER_PARAMS,
    )

    rgb_gif_path = (
        sequence_dir
        / "rgb_motion.gif"
    )

    frames_to_gif(
        frames_dir=seq["frames_dir"],
        output_gif=rgb_gif_path,
        fps=30,
        loop=0,
    )

    ev = frames_to_events_npy_and_gif(
        frames_dir=seq["frames_dir"],
        output_dir=events_dir,
        **EVENT_PARAMS,
    )

    return {
        "name": object_name,
        "object_path": object_path,
        "sequence_dir": sequence_dir,
        "frames_dir": Path(
            seq["frames_dir"]
        ),
        "rgb_gif_path": rgb_gif_path,
        "events_dir": events_dir,
        "events_npy": Path(
            ev["npy_path"]
        ),
        "events_gif_path": Path(
            ev["gif_path"]
        ),
        "events_dat": Path(
            ev["dat_path"]
        ),
    }


object_runs = [
    prepare_single_object_stream(
        spec,
        seed=0,
    )
    for spec in OBJECT_SPECS
]

print(
    f"Prepared "
    f"{len(object_runs)} "
    f"object streams."
)

In [ ]:
def load_existing_object_stream(spec):
    object_name = spec["name"]
    object_path = repo / spec["relative_path"]

    obj_root = OUTPUT_ROOT / object_name
    sequence_dir = obj_root / "motion_sequence"

    frames_dir = sequence_dir / "frames"
    rgb_gif_path = sequence_dir / "rgb_motion.gif"

    events_dir = sequence_dir / "events"
    events_npy = events_dir / "events.npy"
    events_gif_path = events_dir / "events.gif"
    events_dat = events_dir / "events.dat"

    required = [
        frames_dir,
        rgb_gif_path,
        events_npy,
        events_gif_path,
    ]

    missing = [p for p in required if not p.exists()]

    if missing:
        raise FileNotFoundError(
            f"Missing existing outputs for {object_name}:\n"
            + "\n".join(f"  {p}" for p in missing)
        )

    return {
        "name": object_name,
        "object_path": object_path,
        "sequence_dir": sequence_dir,
        "frames_dir": frames_dir,
        "rgb_gif_path": rgb_gif_path,
        "events_dir": events_dir,
        "events_npy": events_npy,
        "events_gif_path": events_gif_path,
        "events_dat": events_dat,
    }


object_runs = [
    load_existing_object_stream(spec)
    for spec in OBJECT_SPECS
]

print(f"Loaded {len(object_runs)} existing object streams.")

for run in object_runs:
    print(
        f"{run['name']:10s} -> "
        f"RGB: {run['rgb_gif_path'].exists()} | "
        f"DVS: {run['events_gif_path'].exists()}"
    )

In [ ]:
rgb_paths = [run["rgb_gif_path"] for run in object_runs]
rgb_titles = [run["name"] for run in object_runs]

display_gif_grid(
    rgb_paths,
    rgb_titles,
    ncols=3,
    cell_width=320,
)

In [ ]:
dvs_paths = [run["events_gif_path"] for run in object_runs]
dvs_titles = [run["name"] for run in object_runs]

display_gif_grid(
    dvs_paths,
    dvs_titles,
    ncols=3,
    cell_width=320,
)

In [ ]:
out = plot_rgb_dvs_snapshot_grid(
    object_runs=object_runs,
    rgb_frame_idx=20,
    resolution=(RESOLUTION, RESOLUTION),
    fps=FPS,
    dvs_window_us=EVENT_PARAMS["gif_window_us"],
    figures_dir="figures",
    save_png=True,
    save_pdf=True,
    save_svg=True,
    show=True,
)

## 3. Proto-object saliency and fixation-selection policy

As in the previous tutorial, the attention module processes the event stream in short temporal windows. For each window, events are accumulated into a 2D event frame and transformed into a **proto-object saliency map** using a multiscale centre-surround computation. The resulting saliency map $S(x,y)$ highlights spatially coherent regions with strong local event structure, providing a bottom-up estimate of where informative object features are likely to be located.

Importantly, **saliency computation and fixation selection are separate operations**. Throughout this notebook, the event input and saliency computation are kept fixed. We modify only the policy used to convert the saliency map into the next fixation location.

- **Softmax sampling.** Under the stochastic policy, the next fixation is sampled from a probability distribution over the saliency map: $P(i \mid S,\beta) = \frac{\exp(\beta S_i)}{\sum_j \exp(\beta S_j)}$, where $S_i$ is the normalised saliency at location $i$ and $\beta$ is an inverse-temperature parameter controlling how strongly saliency biases fixation selection. In other words, $\beta$ regulates the balance between broad exploration and focused exploitation.

- **Argmax selection.** As a deterministic winner-take-all control, the next fixation is selected directly from the maximally salient location: $i^\star = \arg\max_i S_i$. Argmax therefore uses the same event stream and the same proto-object saliency map as Softmax, but removes stochastic exploration entirely, corresponding to the purely exploitative limit. Consistently, as $\beta \rightarrow \infty$, the Softmax distribution becomes increasingly concentrated at the maximally salient location and approaches Argmax selection.

In [ ]:
out = plot_attentional_gain_concept(
    figures_dir="figures",
    save_png=True,
    save_pdf=True,
    save_svg=True,
    show=True,
)

## 4. Experiments

The conceptual figure above illustrates how Softmax gain controls the exploration--exploitation balance. We now quantify this behaviour on real DVS streams using two complementary readouts:

1. **Explored object area** — a proxy for how much of the event-defined object is visited by attention. To obtain this measure, we divide the visual field into a regular spatial grid and use the accumulated DVS activity to determine which cells are likely to belong to the object. Specifically, a grid cell is labelled as an **object cell** when the total number of events falling within that cell exceeds a predefined threshold. Thus, the object is defined directly from its event activity rather than from a ground-truth segmentation mask. A cell is then considered **visited** if at least one fixation falls within it. :contentReference[oaicite:0]{index=0}

   The explored object area is therefore computed as

   $$
   A_{\mathrm{explored}}
   =
   \frac{N_{\mathrm{visited\ object\ cells}}}
   {N_{\mathrm{object\ cells}}}.
   $$

   This quantity should be interpreted as an **event-based proxy for spatial object coverage**, rather than as the fraction of the object's true geometric area that has been observed. A value close to $1$ indicates that attention has visited most regions exhibiting substantial object-related event activity, whereas a low value indicates that fixations remain concentrated within a small part of that event-defined region. In the current implementation, the grid-cell size is set relative to the image dimensions and cells containing at least `noise_thresh` accumulated events are treated as object cells.

2. **Normalised fixation entropy** — a complementary measure of how evenly attention is distributed across the detected object cells. Whereas explored object area asks **how many different object regions are reached**, fixation entropy asks **how evenly fixations are allocated among them**. For each object cell $k$, we compute the fraction of fixations assigned to that cell, $p_k$. The fixation entropy is then $H = -\sum_k p_k \log p_k$. A low entropy indicates that fixations are concentrated within a small number of object cells, whereas a high entropy indicates that attention is distributed more broadly across the object. Because the maximum possible entropy depends on the number of detected object cells $K$, we report the normalised entropy $H_{\mathrm{norm}} = \frac{H}{\log K}$. This maps the measure onto a comparable scale across objects with different numbers of event-defined cells: values near $0$ indicate strongly concentrated sampling, while values near $1$ indicate a nearly uniform distribution of fixations across the available object cells.

Together, these two measures capture related but distinct aspects of attentional exploration. **Explored object area measures spatial coverage**, whereas **normalised fixation entropy measures the diversity of sampling within that space**.



In [ ]:
def run_attention_once(
    events_npy,
    out_dir,
    mode="softmax",
    beta=None,
    seed=0,
):
    params = dict(
        ATTENTION_PARAMS_BASE
    )

    params["mode"] = mode
    params["seed"] = int(seed)

    if mode == "softmax":
        if beta is None:
            raise ValueError(
                "Softmax mode requires beta."
            )

        params["beta"] = float(beta)

    elif mode == "argmax":
        params.pop(
            "beta",
            None,
        )

    else:
        raise ValueError(
            f"Unknown attention mode: {mode}"
        )

    out_dir = Path(out_dir)

    out_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    return run_attention(
        events_npy=events_npy,
        output_dir=out_dir,
        resolution=(
            RESOLUTION,
            RESOLUTION,
        ),
        window_period_ms=WINDOW_PERIOD_MS,
        max_windows=None,
        sigma=None,
        use_polarity=False,
        clear_existing=True,
        attention_params=params,

        # No plotting during experimental sweeps
        plot=False,
        plot_gif_path=None,
        plot_fps=10,
        plot_loop=0,
    )

## 4.1. Softmax vs. Argmax
The first experiment asks whether stochastic Softmax sampling explores the object more effectively than deterministic Argmax selection.

In [ ]:
softmax_argmax_rows = []

for run in object_runs:

    object_name = run["name"]
    events_npy = run["events_npy"]

    # ============================================================
    # Softmax: 5 stochastic seeds
    # ============================================================
    for seed in SEEDS:

        print(
            f"{object_name} | "
            f"softmax | "
            f"beta={SOFTMAX_COMPARE_BETA} | "
            f"seed={seed}"
        )

        out_dir = (
            run["sequence_dir"]
            / "attention_experiments"
            / "softmax_vs_argmax"
            / f"softmax_beta_{SOFTMAX_COMPARE_BETA}"
            / f"seed_{seed}"
        )

        att = run_attention_once(
            events_npy=events_npy,
            out_dir=out_dir,
            mode="softmax",
            beta=SOFTMAX_COMPARE_BETA,
            seed=seed,
        )

        metrics = (
            compute_attention_exploration_metrics(
                events_npy=events_npy,
                saccades_path=att["saccades_path"],
                resolution=(
                    RESOLUTION,
                    RESOLUTION,
                ),
                per=GRID_PER,
                noise_thresh=NOISE_THRESH,
                exclude_initial_fixation=(
                    EXCLUDE_INITIAL_FIXATION
                ),
            )
        )

        softmax_argmax_rows.append({
            "object": object_name,
            "mode": "softmax",
            "beta": SOFTMAX_COMPARE_BETA,
            "seed": seed,
            **metrics,
        })

    # ============================================================
    # Argmax: deterministic control
    # ============================================================
    print(
        f"{object_name} | argmax"
    )

    out_dir = (
        run["sequence_dir"]
        / "attention_experiments"
        / "softmax_vs_argmax"
        / "argmax"
    )

    att = run_attention_once(
        events_npy=events_npy,
        out_dir=out_dir,
        mode="argmax",
        beta=None,
        seed=0,
    )

    metrics = (
        compute_attention_exploration_metrics(
            events_npy=events_npy,
            saccades_path=att["saccades_path"],
            resolution=(
                RESOLUTION,
                RESOLUTION,
            ),
            per=GRID_PER,
            noise_thresh=NOISE_THRESH,
            exclude_initial_fixation=(
                EXCLUDE_INITIAL_FIXATION
            ),
        )
    )

    softmax_argmax_rows.append({
        "object": object_name,
        "mode": "argmax",
        "beta": np.nan,
        "seed": 0,
        **metrics,
    })


softmax_argmax_df = pd.DataFrame(
    softmax_argmax_rows
)

softmax_argmax_df.to_csv(
    OUTPUT_ROOT
    / "softmax_vs_argmax_metrics.csv",
    index=False,
)

softmax_argmax_df

In [ ]:
softmax_argmax_object_df = (
    softmax_argmax_df
    .groupby(
        [
            "object",
            "mode",
        ],
        as_index=False,
    )
    .agg(
        area_explored_coeff=(
            "area_explored_coeff",
            "mean",
        ),
        fixation_entropy_norm=(
            "fixation_entropy_norm",
            "mean",
        ),
        fixation_entropy=(
            "fixation_entropy",
            "mean",
        ),
        num_fixations=(
            "num_fixations",
            "mean",
        ),
        num_fixations_on_object=(
            "num_fixations_on_object",
            "mean",
        ),
    )
)

softmax_argmax_object_df

In [ ]:
from analysis_helpers import (
    plot_softmax_argmax_by_object,
)

out = plot_softmax_argmax_by_object(
    softmax_argmax_df=softmax_argmax_df,
    softmax_argmax_object_df=(
        softmax_argmax_object_df
    ),
    objects_order=objects_order,
    figures_dir="figures",
    save_png=True,
    save_pdf=True,
    save_svg=True,
)

In [ ]:
from analysis_helpers import (
    plot_softmax_argmax_paired,
)

out = plot_softmax_argmax_paired(
    softmax_argmax_object_df=(
        softmax_argmax_object_df
    ),
    objects_order=objects_order,
    softmax_beta=SOFTMAX_COMPARE_BETA,
    figures_dir="figures",
    save_png=True,
    save_pdf=True,
    save_svg=True,
)

## 4.2. Softmax gain sweep
The second asks whether exploration simply continues to increase as Softmax sampling becomes more diffuse, or whether an **intermediate gain** produces a more effective balance between exploring new object regions and exploiting salient structure.

In [ ]:
beta_rows = []

for run in object_runs:

    object_name = run["name"]
    events_npy = run["events_npy"]

    for beta in BETA_SWEEP:

        beta_token = str(
            beta
        ).replace(
            ".",
            "p",
        )

        for seed in SEEDS:

            print(
                f"{object_name} | "
                f"beta={beta} | "
                f"seed={seed}"
            )

            out_dir = (
                run["sequence_dir"]
                / "attention_experiments"
                / "beta_sweep"
                / f"beta_{beta_token}"
                / f"seed_{seed}"
            )

            att = run_attention_once(
                events_npy=events_npy,
                out_dir=out_dir,
                mode="softmax",
                beta=beta,
                seed=seed,
            )

            metrics = (
                compute_attention_exploration_metrics(
                    events_npy=events_npy,
                    saccades_path=att[
                        "saccades_path"
                    ],
                    resolution=(
                        RESOLUTION,
                        RESOLUTION,
                    ),
                    per=GRID_PER,
                    noise_thresh=NOISE_THRESH,
                    exclude_initial_fixation=(
                        EXCLUDE_INITIAL_FIXATION
                    ),
                )
            )

            beta_rows.append({
                "object": object_name,
                "beta": beta,
                "seed": seed,
                **metrics,
            })


beta_df = pd.DataFrame(
    beta_rows
)

beta_df.to_csv(
    OUTPUT_ROOT
    / "beta_sweep_metrics.csv",
    index=False,
)

beta_df

In [ ]:
beta_object_df = (
    beta_df
    .groupby(
        [
            "object",
            "beta",
        ],
        as_index=False,
    )
    .agg(
        area_explored_coeff=(
            "area_explored_coeff",
            "mean",
        ),
        fixation_entropy_norm=(
            "fixation_entropy_norm",
            "mean",
        ),
        fixation_entropy=(
            "fixation_entropy",
            "mean",
        ),
        num_fixations=(
            "num_fixations",
            "mean",
        ),
        num_fixations_on_object=(
            "num_fixations_on_object",
            "mean",
        ),
    )
)

beta_object_df.to_csv(
    OUTPUT_ROOT
    / "beta_sweep_object_means.csv",
    index=False,
)

beta_object_df

In [ ]:
from analysis_helpers import (
    plot_exploration_balance_exploitation_paired,
)

out = plot_exploration_balance_exploitation_paired(
    beta_object_df=beta_object_df,
    softmax_argmax_object_df=(
        softmax_argmax_object_df
    ),

    exploration_beta=0.5,
    balance_beta=5.0,

    objects_order=objects_order,
    figures_dir="figures",
    save_png=True,
    save_pdf=True,
    save_svg=True,
)